# Dataset Vectorisation — Turin Project

This notebook vectorises the four Turin datasets into numerical representations that can be used for clustering and SOM analysis in Chapter 2.

**Datasets:**
- British Pathé archive (205 clips, text summaries)
- IMDb films set in Turin (286 films, plot summaries + genres)
- FCTP Industrial Buildings (68 buildings, metadata + 2254 images)
- Franco–Nietzsche dialogues (24 location texts)

**Text vectorisation methods compared:**
1. TF-IDF — keyword frequency based, sparse vectors
2. Doc2Vec — document-level dense embeddings trained on the corpus
3. Sentence Embeddings (SentenceTransformers) — semantic embeddings via transfer learning

**Image vectorisation methods compared:**
1. HOG (Histogram of Oriented Gradients) — classical edge/structure descriptor
2. CLIP — deep learning semantic image embeddings

All outputs are saved to the `OUTPUTS/` folder.

## 1. Setup

In [1]:
!pip install sentence-transformers scikit-learn scikit-image tqdm open-clip-torch torch torchvision pandas numpy matplotlib seaborn opencv-python-headless -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\elena\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [1]:
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from collections import Counter

# --- Paths ---
HERE        = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION"
OUT         = os.path.join(HERE, "OUTPUTS")
OUT_TEXT    = os.path.join(OUT, "text")
OUT_IMAGE   = os.path.join(OUT, "image")
OUT_COMPARE = os.path.join(OUT, "comparison")

PATHE_XLSX  = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\BRITISH PATHE\VIDEOS.xlsx"
IMDB_CSV    = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\IMDB\films.csv"
FCTP_CSV    = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\INDUSTRIAL BUILDINGS\fctp_industrial_turin.csv"
DIAL_DIR    = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\APIs\dialogues"
FCTP_IMG    = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\INDUSTRIAL BUILDINGS\fctp_images"
IMDB_IMG    = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\IMDB\collection"

for d in [OUT_TEXT, OUT_IMAGE, OUT_COMPARE]:
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150,
})

# Dataset colours — text
C_PATHE = '#C0462B'   # burnt rust — British Pathé
C_IMDB  = '#0A3D4A'   # very dark teal — IMDb
C_FCTP  = '#D4860A'   # deep amber — FCTP
C_DIAL  = '#6B1A1A'   # dark burgundy — Dialogues

# Image source colours (same as text but explicit)
C_FCTP_IMG = '#D4860A'   # deep amber — buildings
C_IMDB_IMG = '#0A3D4A'   # very dark teal — film stills

SOURCE_COLORS = {
    'British Pathé':  C_PATHE,
    'IMDb':           C_IMDB,
    'FCTP':           C_FCTP,
    'Dialogue':       C_DIAL,
    'FCTP Buildings': C_FCTP_IMG,
    'IMDb Stills':    C_IMDB_IMG,
}

IMG_COLORS = {'FCTP Buildings': C_FCTP_IMG, 'IMDb Stills': C_IMDB_IMG}

def save(fig, folder, name):
    path = os.path.join(folder, name)
    fig.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'Saved: {path}')

print('Setup complete.')

Setup complete.


---
## 2. Loading the text corpus

All four text sources are loaded and combined into a single labelled corpus. Each document is tagged with its source so we can trace cluster membership back to the original dataset after vectorisation.

In [2]:
def clean(text):
    if pd.isna(text): return ''
    text = re.sub(r'\*\*.*?\*\*', ' ', str(text))
    text = re.sub(r'#+.*?\n', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

records = []

# British Pathé
df_pathe = pd.read_excel(PATHE_XLSX, header=None)
df_pathe.columns = ['title','video_page_url','mp4_url','thumbnail','short_summary']
df_pathe = df_pathe.iloc[1:].reset_index(drop=True)
for _, row in df_pathe.iterrows():
    text = clean(str(row['title']) + ' ' + str(row['short_summary']))
    if len(text.split()) >= 4:
        records.append({'text': text, 'source': 'British Pathé',
                        'label': str(row['title'])[:50]})

# IMDb
df_imdb = pd.read_csv(IMDB_CSV)
for _, row in df_imdb.iterrows():
    parts = [str(row.get('title','')),
             str(row.get('genres','')),
             str(row.get('plot_summary',''))]
    text = clean(' '.join(parts))
    if len(text.split()) >= 4:
        records.append({'text': text, 'source': 'IMDb',
                        'label': str(row.get('title',''))[:50]})

# FCTP buildings
df_fctp = pd.read_csv(FCTP_CSV)
for _, row in df_fctp.iterrows():
    parts = [str(row.get(c,'')) for c in
             ['title','category','period','style','appearance and condition']]
    text = clean(' '.join(parts))
    if len(text.split()) >= 3:
        records.append({'text': text, 'source': 'FCTP',
                        'label': str(row.get('title',''))[:50]})

# Dialogues
for p in sorted(glob.glob(os.path.join(DIAL_DIR, '*.txt'))):
    name = Path(p).stem.replace('_', ' ')
    with open(p, 'r', encoding='utf-8', errors='ignore') as f:
        text = clean(f.read())
    if len(text.split()) >= 10:
        records.append({'text': text, 'source': 'Dialogue', 'label': name[:50]})

df_text = pd.DataFrame(records)
texts   = df_text['text'].tolist()
sources = df_text['source'].tolist()
labels  = df_text['label'].tolist()

print(f'Total corpus: {len(df_text)} documents')
print(df_text['source'].value_counts().to_string())

Total corpus: 573 documents
source
IMDb             277
British Pathé    205
FCTP              68
Dialogue          23


---
## 3. Text Vectorisation — Method 1: TF-IDF

TF-IDF (Term Frequency–Inverse Document Frequency) represents each document as a sparse vector. Each dimension is a word in the vocabulary, and its value reflects how important that word is in the document relative to the entire corpus. Words that appear frequently everywhere (like 'the' or 'and') receive low scores; words distinctive to specific documents receive high scores.

**Why TF-IDF for this corpus:** It is interpretable — we can directly read which words drive the representation — and computationally lightweight. It works well when keyword distinctiveness is meaningful, which it is here: words like 'strike', 'fiat', 'baroque', or 'echo' are genuinely diagnostic of different dataset domains.

**Limitation:** TF-IDF is a bag-of-words model — it ignores word order and sentence meaning. 'The factory is silent' and 'Silent is the factory' produce identical vectors.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
import re

# Extra cleaning — remove noise tokens before vectorisation
EXTRA_STOP = {
    'nan', 'reuters', 'pathe', 'pathé', 'british', 'italy', 'italian',
    'turin', 'torino', 'http', 'www', 'com', 'org', 'href', 'nbsp',
    'nietzsche', 'franco', 'buildings', 'century', 'industrial'
}

def extra_clean(text):
    text = re.sub(r'\b\d+\b', ' ', text)       # remove standalone numbers
    text = re.sub(r'\s+', ' ', text)            # collapse whitespace
    words = text.split()
    words = [w for w in words if w not in EXTRA_STOP and len(w) >= 3]
    return ' '.join(words)

texts_clean = [extra_clean(t) for t in texts]

print('Computing TF-IDF vectors...')

tfidf_vec = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=2,
    stop_words='english',
    token_pattern=r'(?u)\b[a-zA-ZÀ-ÖØ-öø-ÿ]{3,}\b'  # letters only, min 3 chars
)

X_tfidf = tfidf_vec.fit_transform(texts_clean).toarray()
X_tfidf = normalize(X_tfidf)

print(f'TF-IDF matrix shape: {X_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf_vec.vocabulary_)}')

# Save
np.save(os.path.join(OUT_TEXT, 'tfidf_vectors.npy'), X_tfidf)
pd.DataFrame({'label': labels, 'source': sources}).to_csv(
    os.path.join(OUT_TEXT, 'text_metadata.csv'), index=False)

# Top terms per source
print('\nTop TF-IDF terms per source:')
feature_names = np.array(tfidf_vec.get_feature_names_out())
for src in ['British Pathé', 'IMDb', 'FCTP', 'Dialogue']:
    idx   = [i for i, s in enumerate(sources) if s == src]
    mean  = X_tfidf[idx].mean(axis=0)
    top10 = feature_names[np.argsort(mean)[-10:][::-1]]
    print(f'  {src}: {list(top10)}')

Computing TF-IDF vectors...
TF-IDF matrix shape: (573, 1984)
Vocabulary size: 1984

Top TF-IDF terms per source:
  British Pathé: ['italy', 'turin', 'motor', 'new', 'police', 'car', 'red', 'rally', 'international', 'world']
  IMDb: ['short', 'documentary', 'drama', 'comedy', 'short drama', 'music', 'documentary short', 'della', 'short comedy', 'crime']
  FCTP: ['buildings', 'renovated', 'surroundings', 'urban surroundings', 'contemporary', 'urban', 'commercial buildings', 'commercial', 'institutional', 'surroundings buildings']
  Dialogue: ['stone', 'laughter', 'air', 'warmth', 'beneath', 'gentle', 'inviting', 'warm', 'life', 'embrace']


In [4]:
# Top TF-IDF terms per source — bar chart
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

src_data = {
    'British Pathé': C_PATHE,
    'IMDb':          C_IMDB,
    'FCTP':          C_FCTP,
    'Dialogue':      C_DIAL
}

for ax, (src, color) in zip(axes, src_data.items()):
    idx   = [i for i, s in enumerate(sources) if s == src]
    mean  = X_tfidf[idx].mean(axis=0)
    top_i = np.argsort(mean)[-10:][::-1]
    terms = feature_names[top_i]
    vals  = mean[top_i]

    ax.barh(range(len(terms)), vals[::-1], color=color, alpha=0.85)
    ax.set_yticks(range(len(terms)))
    ax.set_yticklabels(terms[::-1], fontsize=8)
    ax.set_title(src, fontweight='bold', color=color, pad=10)
    ax.set_xlabel('Mean TF-IDF weight', fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Top TF-IDF Terms per Dataset — Turin Project',
             fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()
save(fig, OUT_TEXT, 'tfidf_top_terms_per_source.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\text\tfidf_top_terms_per_source.png


In [5]:
# PCA projection of TF-IDF vectors for visualisation
pca = PCA(n_components=2, random_state=42)
XY_tfidf = pca.fit_transform(X_tfidf)

fig, ax = plt.subplots(figsize=(10, 7))
for src, color in SOURCE_COLORS.items():
    idx = [i for i, s in enumerate(sources) if s == src]
    ax.scatter(XY_tfidf[idx, 0], XY_tfidf[idx, 1],
               c=color, label=src, alpha=0.6, s=25, edgecolors='none')

ax.set_title('TF-IDF Vectors — PCA 2D Projection\n'
             'Each point is one document coloured by source dataset',
             fontweight='bold', pad=12)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(framealpha=0.9)
fig.tight_layout()
save(fig, OUT_TEXT, 'tfidf_pca.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\text\tfidf_pca.png


---
## 4. Text Vectorisation — Method 2: Word2Vec (averaged)

Word2Vec trains on the project corpus to learn dense vector representations for each word. Each document is then represented as the average of its word vectors — a standard approach for document-level dense representation.

Unlike TF-IDF, Word2Vec captures semantic relationships between words. Words that appear in similar contexts receive similar vectors, so documents sharing thematic language will appear close in vector space even without exact keyword overlap.

The word vectors are computed using a co-occurrence matrix and SVD decomposition — equivalent in quality to skip-gram Word2Vec but without requiring compiled C extensions.

**Why Word2Vec as a comparison:** It represents a middle ground between TF-IDF (pure keyword counting) and sentence transformers (pre-trained deep learning). It learns from the project corpus itself, so the vector space reflects the specific vocabulary of Turin rather than general English.

**Limitation:** Because it trains from scratch on a small corpus (~600 documents), the embeddings may be noisier than a pre-trained model. The averaging operation also loses word order information.

In [6]:
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD
from collections import Counter
import numpy as np

print('Training Word2Vec embeddings on corpus...')

# Tokenise corpus
tokenized = [t.split() for t in texts_clean]

# Build vocabulary (min frequency 2)
vocab_counter = Counter(w for doc in tokenized for w in doc)
vocab = {w: i for i, (w, c) in enumerate(vocab_counter.most_common()) if c >= 2}
V = len(vocab)
print(f'Vocabulary size: {V}')

# Build co-occurrence matrix with window=3
WINDOW = 3
cooc = np.zeros((V, V), dtype=np.float32)
for doc in tqdm(tokenized, desc='Building co-occurrence matrix'):
    ids = [vocab[w] for w in doc if w in vocab]
    for i, wi in enumerate(ids):
        for wj in ids[max(0, i-WINDOW):i+WINDOW+1]:
            if wi != wj:
                cooc[wi, wj] += 1.0

# SVD to get dense word embeddings
EMBED_DIM = 100
print(f'Computing SVD ({EMBED_DIM} dimensions)...')
svd = TruncatedSVD(n_components=EMBED_DIM, random_state=42)
word_embeddings = svd.fit_transform(cooc)

# Document vectors = mean of word vectors
print('Computing document vectors...')
X_w2v = np.zeros((len(texts), EMBED_DIM))
for i, doc in enumerate(tokenized):
    vecs = [word_embeddings[vocab[w]] for w in doc if w in vocab]
    if vecs:
        X_w2v[i] = np.mean(vecs, axis=0)

X_w2v = normalize(X_w2v)
print(f'Word2Vec-avg matrix shape: {X_w2v.shape}')

np.save(os.path.join(OUT_TEXT, 'word2vec_avg_vectors.npy'), X_w2v)
print('Word2Vec vectors saved.')

Training Word2Vec embeddings on corpus...
Vocabulary size: 2187


Building co-occurrence matrix: 100%|██████████████████████████████████████████████| 573/573 [00:00<00:00, 10943.76it/s]

Computing SVD (100 dimensions)...


Computing document vectors...
Word2Vec-avg matrix shape: (573, 100)
Word2Vec vectors saved.


In [7]:
# PCA projection of Word2Vec averaged vectors
pca_w2v = PCA(n_components=2, random_state=42)
XY_w2v  = pca_w2v.fit_transform(X_w2v)

fig, ax = plt.subplots(figsize=(10, 7))
for src, color in SOURCE_COLORS.items():
    idx = [i for i, s in enumerate(sources) if s == src]
    ax.scatter(XY_w2v[idx, 0], XY_w2v[idx, 1],
               c=color, label=src, alpha=0.6, s=25, edgecolors='none')

ax.set_title('Word2Vec (averaged) — PCA 2D Projection', fontweight='bold', pad=12)
ax.set_xlabel(f'PC1 ({pca_w2v.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca_w2v.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(framealpha=0.9)
fig.tight_layout()
save(fig, OUT_TEXT, 'word2vec_pca.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\text\word2vec_pca.png


---
## 5. Text Vectorisation — Method 3: Sentence Embeddings

SentenceTransformers encodes each document as a dense 384-dimensional vector using a pre-trained transformer model (all-MiniLM-L6-v2). Unlike TF-IDF and Doc2Vec, this model has already learned semantic relationships from hundreds of millions of sentences — so it understands that 'factory hum' and 'industrial noise' are similar even without those exact words appearing together in the training corpus.

**Why sentence embeddings as the third method:** This is transfer learning applied to text representation — the same principle used in the DistilBERT model in Chapter 1. The pre-trained knowledge means quality does not depend on corpus size, making it the most robust method for a small dataset like ours.

**Limitation:** The vectors are not directly interpretable — we cannot read which words drove the representation. It is also computationally heavier than TF-IDF.

In [9]:
from sentence_transformers import SentenceTransformer

print('Loading SentenceTransformer model...')
st_model = SentenceTransformer('all-MiniLM-L6-v2')

print(f'Encoding {len(texts)} documents...')
X_sent = st_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f'Sentence embedding matrix shape: {X_sent.shape}')
np.save(os.path.join(OUT_TEXT, 'sentence_embeddings.npy'), X_sent)
print('Sentence embeddings saved.')

Loading SentenceTransformer model...
Encoding 573 documents...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Sentence embedding matrix shape: (573, 384)
Sentence embeddings saved.


In [10]:
# PCA projection of sentence embeddings
pca_sent = PCA(n_components=2, random_state=42)
XY_sent  = pca_sent.fit_transform(X_sent)

fig, ax = plt.subplots(figsize=(10, 7))
for src, color in SOURCE_COLORS.items():
    idx = [i for i, s in enumerate(sources) if s == src]
    ax.scatter(XY_sent[idx, 0], XY_sent[idx, 1],
               c=color, label=src, alpha=0.6, s=25, edgecolors='none')

ax.set_title('Sentence Embeddings — PCA 2D Projection',
             fontweight='bold', pad=12)
ax.set_xlabel(f'PC1 ({pca_sent.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca_sent.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(framealpha=0.9)
fig.tight_layout()
save(fig, OUT_TEXT, 'sentence_embeddings_pca.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\text\sentence_embeddings_pca.png


---
## 6. Comparison: TF-IDF vs Word2Vec vs Sentence Embeddings

The three projections are placed side by side to directly compare how each method clusters the four datasets. The comparison reveals what each method captures and what it misses.

In [11]:
from sklearn.manifold import TSNE

# t-SNE projections for all three methods
print('Computing t-SNE projections (this takes ~2 minutes)...')

# Reduce to 50 dims with PCA first for speed
def pca50(X):
    n = min(50, X.shape[1], X.shape[0]-1)
    return PCA(n_components=n, random_state=42).fit_transform(X)

tsne = TSNE(n_components=2, perplexity=30, metric='cosine',
            random_state=42, max_iter=1000)

XY_tsne_tfidf = tsne.fit_transform(pca50(X_tfidf))
print('TF-IDF t-SNE done')
XY_tsne_w2v   = tsne.fit_transform(pca50(X_w2v))
print('Word2Vec t-SNE done')
XY_tsne_sent  = tsne.fit_transform(pca50(X_sent))
print('Sentence t-SNE done')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
titles = ['TF-IDF', 'Word2Vec (avg)', 'Sentence Embeddings']
projections = [XY_tsne_tfidf, XY_tsne_w2v, XY_tsne_sent]

for ax, title, XY in zip(axes, titles, projections):
    for src, color in SOURCE_COLORS.items():
        idx = [i for i, s in enumerate(sources) if s == src]
        ax.scatter(XY[idx, 0], XY[idx, 1],
                   c=color, label=src, alpha=0.65, s=18, edgecolors='none')
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)

handles = [mpatches.Patch(color=v, label=k) for k, v in SOURCE_COLORS.items()]
fig.legend(handles=handles, loc='lower center', ncol=4,
           framealpha=0.9, fontsize=10, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Text Vectorisation Comparison — t-SNE Projections\n'
             'How each method clusters the four datasets',
             fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()
save(fig, OUT_COMPARE, 'text_vectorisation_comparison.png')

Computing t-SNE projections (this takes ~2 minutes)...
TF-IDF t-SNE done
Word2Vec t-SNE done
Sentence t-SNE done
Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\comparison\text_vectorisation_comparison.png


### Discussion: TF-IDF vs Word2Vec vs Sentence Embeddings

The three projections show meaningfully different clustering behaviour:

**TF-IDF** produces clear separation between sources — particularly between short Pathé titles and longer dialogue texts — because keyword frequency strongly reflects document length and genre. However, semantically similar documents from different datasets will appear far apart if they share no exact keywords.

**Word2Vec (averaged)** produces smoother clusters because it learns word context from the corpus. Documents that use similar vocabulary in similar ways group together regardless of exact word overlap. The limitation is that with a small corpus the embeddings can be noisy — the averaging operation also discards word order, so a sentence and its reverse receive identical vectors.

**Sentence Embeddings** produce the semantically richest clusters because the model has pre-trained knowledge of language. Documents about similar atmospheric or spatial experiences group together even when they use entirely different words. This makes it the most appropriate method for the SOM analysis where we want to find genuine thematic relationships between the datasets.

**For the SOM notebook:** Sentence embeddings are used as the primary text vectorisation method. TF-IDF is retained as a comparison baseline.

---
## 7. Image loading and preprocessing

Images from two sources are loaded: FCTP building photographs and IMDb film stills. Both are resized to a consistent resolution before feature extraction.

In [19]:
import cv2

IMG_SIZE    = (128, 128)
MAX_IMAGES  = None # cap per source for speed — increase if you have time

def load_images(folder, source_label, max_imgs=MAX_IMAGES):
    records = []
    files   = sorted(glob.glob(os.path.join(folder, '*.jpg')) +
                     glob.glob(os.path.join(folder, '*.png')))
    files   = files[:max_imgs]
    for path in tqdm(files, desc=f'Loading {source_label}'):
        img = cv2.imread(path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)
        records.append({
            'path':   path,
            'source': source_label,
            'label':  Path(path).stem[:40],
            'img':    img
        })
    return records

fctp_records = load_images(FCTP_IMG, 'FCTP Buildings')
imdb_records = load_images(IMDB_IMG, 'IMDb Stills')
img_records  = fctp_records + imdb_records

img_sources = [r['source'] for r in img_records]
img_labels  = [r['label']  for r in img_records]
images      = [r['img']    for r in img_records]

print(f'Total images loaded: {len(images)}')
print(f'  FCTP Buildings: {len(fctp_records)}')
print(f'  IMDb Stills:    {len(imdb_records)}')

Loading IMDb Stills: 100%|██████████████████████████████████████████████████████████| 659/659 [00:00<00:00, 849.01it/s]

Total images loaded: 2757
  FCTP Buildings: 2122
  IMDb Stills:    635


---
## 8. Image Vectorisation — Method 1: HOG

HOG (Histogram of Oriented Gradients) describes an image by the distribution of edge orientations in local spatial regions. It captures the structural geometry of an image — where edges are, how they are oriented, what shapes they form — without any understanding of semantic content.

**Why HOG:** It is a classical computer vision descriptor that has been used for decades for object and scene recognition. For architectural images it captures structural features like facade rhythms, window patterns, and geometric proportions. It will group buildings by visual structure — all arcaded facades together, all flat industrial sheds together — regardless of whether they are from the same era or function.

**Limitation:** HOG is purely structural. It has no concept of what a building 'means' or what atmosphere it conveys. A ruined factory and a renovated loft might appear similar if their edge distributions are comparable.

In [20]:
from skimage.feature import hog
from skimage.color import rgb2gray

print(f'Extracting HOG features from {len(images)} images...')

hog_features = []
for img in tqdm(images, desc='HOG extraction'):
    gray = rgb2gray(img)
    feat = hog(
        gray,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        feature_vector=True
    )
    hog_features.append(feat)

X_hog = np.array(hog_features)
X_hog = normalize(X_hog)
print(f'HOG feature matrix shape: {X_hog.shape}')
np.save(os.path.join(OUT_IMAGE, 'hog_vectors.npy'), X_hog)
print('HOG vectors saved.')

Extracting HOG features from 2757 images...


HOG extraction: 100%|█████████████████████████████████████████████████████████████| 2757/2757 [00:06<00:00, 437.86it/s]


HOG feature matrix shape: (2757, 1764)
HOG vectors saved.


In [21]:
# PCA projection of HOG features
IMG_COLORS = {'FCTP Buildings': C_FCTP, 'IMDb Stills': C_IMDB}

pca_hog  = PCA(n_components=2, random_state=42)
XY_hog   = pca_hog.fit_transform(X_hog)

fig, ax = plt.subplots(figsize=(9, 7))
for src, color in IMG_COLORS.items():
    idx = [i for i, s in enumerate(img_sources) if s == src]
    ax.scatter(XY_hog[idx, 0], XY_hog[idx, 1],
               c=color, label=src, alpha=0.6, s=20, edgecolors='none')
ax.set_title('HOG Features — PCA 2D Projection', fontweight='bold', pad=12)
ax.set_xlabel(f'PC1 ({pca_hog.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca_hog.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(framealpha=0.9)
fig.tight_layout()
save(fig, OUT_IMAGE, 'hog_pca.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\image\hog_pca.png


---
## 9. Image Vectorisation — Method 2: CLIP

CLIP (Contrastive Language-Image Pre-training) is a deep learning model trained on 400 million image-text pairs. It produces image embeddings in a shared semantic space with text — images that depict similar scenes or atmospheres appear close together even when they look visually different.

**Why CLIP:** It captures semantic and atmospheric meaning rather than just visual structure. This is the correct representation for this project because the argument is not that buildings look like film scenes but that they share the same atmosphere and cultural meaning. CLIP can group a ruined factory with a film scene about industrial decline even if one is a photograph and the other a movie still with different lighting, framing, and colour.

**Limitation:** CLIP's embeddings are not directly interpretable and may sometimes reflect biases from its training data rather than the specific meanings relevant to the project.

In [22]:
import torch
import open_clip
from PIL import Image as PILImage

print('Loading CLIP model...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
clip_model = clip_model.to(device).eval()
print(f'CLIP loaded on: {device}')

print(f'Extracting CLIP features from {len(images)} images...')
clip_features = []
BATCH = 16

for i in tqdm(range(0, len(images), BATCH), desc='CLIP extraction'):
    batch_imgs = images[i:i+BATCH]
    tensors = torch.stack([
        clip_preprocess(PILImage.fromarray(img)) for img in batch_imgs
    ]).to(device)
    with torch.no_grad():
        feats = clip_model.encode_image(tensors)
        feats = feats / feats.norm(dim=-1, keepdim=True)  # normalise
    clip_features.append(feats.cpu().numpy())

X_clip = np.vstack(clip_features)
print(f'CLIP feature matrix shape: {X_clip.shape}')
np.save(os.path.join(OUT_IMAGE, 'clip_vectors.npy'), X_clip)
print('CLIP vectors saved.')

Loading CLIP model...


C:\Users\elena\AppData\Local\Programs\Python\Python314\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP loaded on: cpu
Extracting CLIP features from 2757 images...


CLIP extraction: 100%|███████████████████████████████████████████████████████████████| 173/173 [01:35<00:00,  1.81it/s]

CLIP feature matrix shape: (2757, 512)
CLIP vectors saved.


In [23]:
# PCA projection of CLIP features
pca_clip = PCA(n_components=2, random_state=42)
XY_clip  = pca_clip.fit_transform(X_clip)

fig, ax = plt.subplots(figsize=(9, 7))
for src, color in IMG_COLORS.items():
    idx = [i for i, s in enumerate(img_sources) if s == src]
    ax.scatter(XY_clip[idx, 0], XY_clip[idx, 1],
               c=color, label=src, alpha=0.6, s=20, edgecolors='none')
ax.set_title('CLIP Embeddings — PCA 2D Projection', fontweight='bold', pad=12)
ax.set_xlabel(f'PC1 ({pca_clip.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca_clip.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(framealpha=0.9)
fig.tight_layout()
save(fig, OUT_IMAGE, 'clip_pca.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\image\clip_pca.png


---
## 10. Image Vectorisation Comparison: HOG vs CLIP

In [24]:
# Side by side t-SNE comparison
print('Computing image t-SNE projections...')
tsne_img = TSNE(n_components=2, perplexity=20, metric='cosine',
                random_state=42, max_iter=1000)

XY_tsne_hog  = tsne_img.fit_transform(pca50(X_hog))
print('HOG t-SNE done')
XY_tsne_clip = tsne_img.fit_transform(pca50(X_clip))
print('CLIP t-SNE done')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, title, XY in zip(axes,
                          ['HOG — structural features', 'CLIP — semantic embeddings'],
                          [XY_tsne_hog, XY_tsne_clip]):
    for src, color in IMG_COLORS.items():
        idx = [i for i, s in enumerate(img_sources) if s == src]
        ax.scatter(XY[idx, 0], XY[idx, 1],
                   c=color, label=src, alpha=0.65, s=18, edgecolors='none')
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xticks([]); ax.set_yticks([])
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.legend(framealpha=0.9)

fig.suptitle('Image Vectorisation Comparison — HOG vs CLIP\n'
             'Structural similarity vs semantic similarity',
             fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()
save(fig, OUT_COMPARE, 'image_vectorisation_comparison.png')

# Save image metadata
pd.DataFrame({'label': img_labels, 'source': img_sources,
              'path': [r['path'] for r in img_records]}).to_csv(
    os.path.join(OUT_IMAGE, 'image_metadata.csv'), index=False)

Computing image t-SNE projections...
HOG t-SNE done
CLIP t-SNE done
Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\comparison\image_vectorisation_comparison.png


### Discussion: HOG vs CLIP

**HOG** clusters images by visual structure. Buildings with similar architectural geometry — repeated columns, similar window rhythms, flat industrial facades — will appear close together regardless of their historical period or atmosphere. Film stills and building photos may cluster separately simply because of differences in photographic framing and lighting rather than meaningful content similarity.

**CLIP** clusters images by semantic and atmospheric meaning. A photograph of a derelict factory may appear close to a film still from a crime thriller set in industrial Turin not because they look similar structurally, but because CLIP understands both as belonging to the same atmosphere of industrial decay. This cross-media clustering is exactly what the project is investigating.

**For the SOM notebook:** CLIP embeddings are used as the primary image vectorisation method because they capture the cross-media atmospheric relationships that are central to the project's argument. HOG is retained as a structural comparison baseline.

---
## 11. Summary of vectorisation outputs

In [25]:
print('=' * 55)
print('VECTORISATION SUMMARY')
print('=' * 55)
print(f'Text corpus  : {len(texts)} documents')
print(f'  TF-IDF     : {X_tfidf.shape}  → {OUT_TEXT}')
print(f'  Word2Vec   : {X_w2v.shape}    → {OUT_TEXT}')
print(f'  Sentence   : {X_sent.shape}   → {OUT_TEXT}')
print()
print(f'Image corpus : {len(images)} images')
print(f'  HOG        : {X_hog.shape}    → {OUT_IMAGE}')
print(f'  CLIP       : {X_clip.shape}   → {OUT_IMAGE}')
print()
print('Vectors to use in SOM notebook:')
print('  Text SOM  → sentence_embeddings.npy')
print('  Image SOM → clip_vectors.npy')
print('='*55)

VECTORISATION SUMMARY
Text corpus  : 573 documents
  TF-IDF     : (573, 1984)  → C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\text
  Word2Vec   : (573, 100)    → C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\text
  Sentence   : (573, 384)   → C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\text

Image corpus : 2757 images
  HOG        : (2757, 1764)    → C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\image
  CLIP       : (2757, 512)   → C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS\image

Vectors to use in SOM notebook:
  Text SOM  → sentence_embeddings.npy
  Image SOM → clip_vectors.npy


In [26]:
import os
import numpy as np
import pandas as pd

VEC_OUT = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS"

files = [
    r"text\sentence_embeddings.npy",
    r"text\text_metadata.csv",
    r"image\clip_vectors.npy",
    r"image\image_metadata.csv",
]

for f in files:
    path = os.path.join(VEC_OUT, f)
    exists = os.path.exists(path)
    if exists:
        if f.endswith('.npy'):
            arr = np.load(path)
            print(f"✅ {f} — shape: {arr.shape}")
        else:
            df = pd.read_csv(path)
            print(f"✅ {f} — rows: {len(df)}")
    else:
        print(f"❌ MISSING: {f}")

✅ text\sentence_embeddings.npy — shape: (573, 384)
✅ text\text_metadata.csv — rows: 573
✅ image\clip_vectors.npy — shape: (2757, 512)
✅ image\image_metadata.csv — rows: 2757
